# 01 — Download and clean

Fetches every AEMO public data file the analysis needs, cleans it to parquet, and runs the schema checks.

Runs top to bottom from an empty `data/` directory. The download step is polite (about 6.7 seconds between requests) and skips files already present, so re-running is cheap. A full cold download is roughly 2.7 GB and takes about half an hour.

Checks applied to every interval table:

- 288 dispatch intervals per trading day at five minutes, 48 at thirty
- no duplicate `(interval, facility)` keys — fatal if present, because every revenue total would double-count
- no null values outside rows carrying a `Failure Reason`
- first and last interval match the expected period
- trading day boundary at 08:00 AWST, timestamps parsed day-first and never inferred

Short trading days are listed, never filled.

In [1]:
import sys
from pathlib import Path

import pandas as pd

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from src import clean, data, download_data, download_docs, loss_factors, paths

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
paths.ensure_dirs()
print(f"repository root: {paths.ROOT}")

repository root: C:\GitHub\WA_energy_analysis


## The dataset registry

Each entry records why the analysis needs it. `market-schedule-csv` is the find that makes the FCESS side of this project possible: it is the only public source of enablement quantities by facility.

In [2]:
registry = pd.DataFrame(
    [
        {
            "dataset": d.name,
            "grain": d.grain,
            "local_files": len(list(d.directory.glob("*.csv"))) if d.directory.exists() else 0,
            "purpose": d.purpose,
        }
        for d in download_data.DATASETS
    ]
)
registry

,dataset,grain,local_files,purpose
0,market-clearing-prices-csv,yearly,4,5-minute energy and FCESS clearing prices. The...
1,reference-trading-price-csv,yearly,4,30-minute Reference Trading Price. Used only i...
2,market-schedule-csv,monthly,38,Per-facility energy and FCESS schedule quantit...
3,market-requirements-csv,yearly,4,FCESS requirement quantities per interval. Con...
4,facility-scada-csv,monthly,38,Per-facility metered output every 5 minutes. D...
5,electric-storage-resource-obligation-intervals,yearly,4,ESROI windows a storage facility with Capacity...
6,post-facilities,static,1,"Post-reform facility standing data, including ..."
7,post-participants,static,1,Post-reform participant register. Maps facilit...
8,fcess,static,1,FCESS accreditation standing data: maximum acc...
9,capacity-credits-since-market-start,static,3,Capacity Credits assigned per facility per cap...


## Download

Set `DOWNLOAD = True` to fetch. Files already present are skipped, except the file covering the current period, which AEMO is still appending to.

In [3]:
DOWNLOAD = False

if DOWNLOAD:
    for dataset in download_data.DATASETS:
        download_data.download_dataset(dataset.name)
    download_docs.fetch_all()
else:
    print("skipped; set DOWNLOAD = True for a cold run")
    for dataset in download_data.DATASETS:
        present = len(list(dataset.directory.glob("*.csv"))) if dataset.directory.exists() else 0
        if present == 0:
            print(f"  MISSING {dataset.name}")

skipped; set DOWNLOAD = True for a cold run


## Clean

Writes tidy parquet to `data/processed/`. The two large per-facility datasets are written as one parquet part per month so they never have to be held in memory at once.

In [4]:
summary = clean.run()
summary


prices_5min
    skip (already built; pass --force to rebuild)



prices_30min
    skip (already built; pass --force to rebuild)

market_requirements
    skip (already built; pass --force to rebuild)

esroi
    skip (already built; pass --force to rebuild)

facilities
    skip (already built; pass --force to rebuild)

fcess_accreditation
    skip (already built; pass --force to rebuild)

capacity_credits
    skip (already built; pass --force to rebuild)

reserve_capacity_prices
    skip (already built; pass --force to rebuild)

market_schedule
    skip   MarketSchedule-2023-10.csv


    skip   MarketSchedule-2023-11.csv
    skip   MarketSchedule-2023-12.csv
    skip   MarketSchedule-2024-01.csv
    skip   MarketSchedule-2024-02.csv
    skip   MarketSchedule-2024-03.csv


    skip   MarketSchedule-2024-04.csv
    skip   MarketSchedule-2024-05.csv
    skip   MarketSchedule-2024-06.csv


    skip   MarketSchedule-2024-07.csv
    skip   MarketSchedule-2024-08.csv
    skip   MarketSchedule-2024-09.csv
    skip   MarketSchedule-2024-10.csv


    skip   MarketSchedule-2024-11.csv
    skip   MarketSchedule-2024-12.csv
    skip   MarketSchedule-2025-01.csv
    skip   MarketSchedule-2025-02.csv


... (13 outputs left out)

    skip   FacilityScada-2026-02.csv
    skip   FacilityScada-2026-03.csv


    skip   FacilityScada-2026-04.csv
    skip   FacilityScada-2026-05.csv
    skip   FacilityScada-2026-06.csv
    skip   FacilityScada-2026-07.csv
    skip   FacilityScada-2026-08.csv


    skip   FacilityScada-2026-09.csv
    skip   FacilityScada-2026-10.csv
    skip   FacilityScada-2026-11.csv
facility_scada: 23,393,022 rows
  interval range   2023-10-01 08:00:00  ..  2026-11-02 07:55:00
  trading days     1,128
  duplicate keys   0
  note             written as 38 monthly parquet part(s)


,table,rows,first,last,trading_days,dup_keys,short_days,long_days,failure_rows,null_cols
0,prices_5min,324850,NaT,NaT,0,0,0,0,0,0
1,prices_30min,54144,NaT,NaT,0,0,0,0,0,0
2,market_requirements,324864,NaT,NaT,0,0,0,0,0,0
3,esroi,1465,NaT,NaT,0,0,0,0,0,0
4,facilities,177,NaT,NaT,0,0,0,0,0,0
5,fcess_accreditation,177,NaT,NaT,0,0,0,0,0,0
6,capacity_credits,1177,NaT,NaT,0,0,0,0,0,0
7,reserve_capacity_prices,43,NaT,NaT,0,0,0,0,0,0
8,market_schedule,28405038,2023-10-01 08:00:00,2026-11-02 07:55:00,1128,0,0,0,0,0
9,facility_scada,23393022,2023-10-01 08:00:00,2026-11-02 07:55:00,1128,0,0,0,0,0


## Interval completeness

The five-minute price series should have exactly 288 intervals on every trading day. Days that fall short are listed rather than padded.

In [5]:
data.clear_caches()
prices = data.load_prices()

per_day = prices.groupby("trading_date").size()
short = per_day[per_day != paths.INTERVALS_PER_DAY_5MIN]

print(f"trading days            {len(per_day):,}")
print(f"first / last            {per_day.index.min().date()} .. {per_day.index.max().date()}")
print(f"complete days           {(per_day == paths.INTERVALS_PER_DAY_5MIN).sum():,}")
print(f"short days              {len(short)}")
short.to_frame("intervals")

trading days            1,128


first / last            2023-10-01 .. 2026-11-01
complete days           1,118
short days              10


,intervals
trading_date,
2024-08-23,285
2024-09-03,287
2024-09-10,287
2024-09-25,286
2025-08-27,287
2025-12-02,287
2026-02-11,287
2026-03-12,286
2026-08-18,287


## Trading day boundary

The WEM trading day runs 08:00 to 08:00 AWST. A trading day's first interval must start at 08:00 and its last at 07:55 the following calendar day.

In [6]:
bounds = prices.groupby("trading_date")["interval_start"].agg(["min", "max"])
first_wrong = bounds[bounds["min"].dt.strftime("%H:%M") != "08:00"]
last_wrong = bounds[bounds["max"].dt.strftime("%H:%M") != "07:55"]

print(f"days whose first interval is not 08:00   {len(first_wrong)}")
print(f"days whose last interval is not 07:55    {len(last_wrong)}")
bounds.head(3)

days whose first interval is not 08:00   0
days whose last interval is not 07:55    0


,min,max
trading_date,,
2023-10-01,2023-10-01 08:00:00,2023-10-02 07:55:00
2023-10-02,2023-10-02 08:00:00,2023-10-03 07:55:00
2023-10-03,2023-10-03 08:00:00,2023-10-04 07:55:00


## The settlement price

Energy settles on the thirty-minute Reference Trading Price (ESM Rules 9.9.4), which clause 7.11A.1(b) defines as the time-weighted average of the six five-minute clearing prices in the Trading Interval. Reproducing that average from the five-minute series is a check on both datasets at once.

In [7]:
grouped = prices.groupby("trading_interval")
derived = grouped[data.CLEARING_PRICE_COLUMN].mean()
published = grouped["reference_trading_price"].first()

difference = (derived - published).abs().dropna()
print(f"Trading Intervals compared      {len(difference):,}")
print(f"agree to within one cent        {(difference < 0.01).mean():.4%}")
print(f"largest absolute difference     ${difference.max():,.2f}")
print(f"intervals with no published RTP {published.isna().sum():,}")

Trading Intervals compared      54,144
agree to within one cent        99.9557%
largest absolute difference     $56.45
intervals with no published RTP 0


## Obligation intervals

Assumption D2 reads the published `ESROI Trading Interval End` as the *start* of the final obligation interval, which makes every window eight Trading Intervals — the ESR Obligation Duration in the Rules glossary. If that reading were wrong the windows would come out as seven.

In [8]:
esroi = data.load_esroi()
print("window length in Trading Intervals:")
print(esroi["n_trading_intervals"].value_counts().to_string())

esroi.groupby("capacity_year").agg(
    days=("trading_date", "nunique"),
    window_start=("esroi_start", lambda s: s.mode().iat[0]),
    window_end=("esroi_end", lambda s: s.mode().iat[0]),
    per_component_rows=("market_wide", lambda s: int((~s).sum())),
)

window length in Trading Intervals:
n_trading_intervals
8    1465


,days,window_start,window_end,per_component_rows
capacity_year,,,,
2023-24,366,16:30,20:00,0
2024-25,365,17:30,21:00,0
2025-26,365,17:30,21:00,0
2026-27,41,17:30,21:00,369


## The SCADA unit question

`facility-scada-csv` publishes a column called `Average MWh`, which could be an average MW over the interval or an energy in MWh per interval. Reading it wrongly scales every backcast revenue figure by twelve, so it is settled against registered capacity rather than assumed.

In [9]:
diagnosis = clean.diagnose_scada_unit(sample_months=6, top_n=15)
diagnosis[["registered_mw", "observed_peak", "ratio_to_mw", "ratio_to_mwh_per_interval", "verdict"]]

,registered_mw,observed_peak,ratio_to_mw,ratio_to_mwh_per_interval,verdict
facility_code,,,,,
NEWGEN_NEERABUP_GT1,342.0,28.639,0.083740,1.004877,MWh per interval
NEWGEN_KWINANA_CCG1,334.8,28.317,0.084579,1.014946,MWh per interval
COLLIE_G1,318.3,26.979,0.084760,1.017116,MWh per interval
WARRADARGE_WF1,283.0,14.902,0.052657,0.631887,MWh per interval
COCKBURN_CCG1,240.0,20.840,0.086833,1.042000,MWh per interval
INVESTEC_COLLGAR_WF1,218.5,17.847,0.081680,0.980156,MWh per interval
BW2_BLUEWATERS_G1,217.0,18.369,0.084650,1.015797,MWh per interval
BW1_BLUEWATERS_G2,217.0,18.358,0.084599,1.015189,MWh per interval
MUJA_G7,212.6,17.977,0.084558,1.014694,MWh per interval


The ratio against registered MW / 12 sits at 1.00 for the large thermal units, and the ratio against registered MW sits at 0.083, which is 1/12. The column is **MWh per five-minute interval**. Recorded as assumption D5.

## The storage register

There is no technology column in the public standing data, so storage is identified by facility code and cross-checked against the FCESS accreditation and Capacity Credit registers.

In [10]:
storage = data.storage_facilities()
storage[
    [
        "facility_code",
        "participant_name",
        "registered_mw",
        "max_contingency_raise_mw",
        "max_regulation_raise_mw",
        "max_rocof_mw",
        "speed_factor",
    ]
]

,facility_code,participant_name,registered_mw,max_contingency_raise_mw,max_regulation_raise_mw,max_rocof_mw,speed_factor
0,ALINTA_WGP_ESR1,Alinta Sales Pty Ltd,200.0,NaN,NaN,NaN,NaN
1,COLLIE_BESS2,Collie Battery Pty Ltd ATF Collie Battery Trust,600.0,300.0,200.0,NaN,0.5
2,COLLIE_ESR1,Collie Battery Pty Ltd ATF Collie Battery Trust,400.0,200.0,200.0,NaN,1.0
3,COLLIE_ESR4,Synergy,500.0,240.6,NaN,NaN,0.5
4,COLLIE_ESR5,Synergy,500.0,240.6,NaN,NaN,0.5
5,KWINANA_ESR1,Synergy,200.0,50.0,100.0,NaN,0.5
6,KWINANA_ESR2,Synergy,450.0,199.7,200.0,NaN,0.5
7,MERREDIN_ESR1,Merredin Big Battery Nominee Pty Ltd as truste...,200.0,NaN,NaN,NaN,NaN


## Loss factors

Loss factors are not in the public data file portal, so they are parsed from the annual WEM Loss Factor Reports. Transmission-connected points take a Distribution Loss Factor of 1 (ESM Rules 2.27.4(c)), so each battery's Loss Factor is its Transmission Loss Factor.

In [11]:
factors = loss_factors.load()
factors.pivot(index="facility_code", columns="financial_year", values="loss_factor")

financial_year,2023-24,2024-25,2025-26,2026-27
facility_code,,,,
ALINTA_WGP_ESR1,0.9644,0.9871,0.9559,0.9593
COLLIE_BESS2,0.9698,0.9719,0.9873,0.9901
COLLIE_ESR1,0.9698,0.9719,0.9873,1.0006
COLLIE_ESR4,0.9698,0.9719,0.9732,1.0006
COLLIE_ESR5,0.9698,0.9719,0.9732,1.0006
KWINANA_ESR1,0.9918,0.9983,0.9968,1.0004
KWINANA_ESR2,0.9918,0.9983,0.9978,1.0010
MERREDIN_ESR1,1.0337,1.0641,1.0390,1.0444
MUCHEA_ESR1,1.0008,1.0041,1.0066,1.0072


## Reserve Capacity Prices

Taken as published rather than derived from the rule formula, which depends on the Benchmark Reserve Capacity Price and on a regime that changed within the period (assumption C5).

In [12]:
rcp = data.load_reserve_capacity_prices()
rcp[rcp["price_type_canonical"] == "Reserve Capacity"][
    ["capacity_year", "price_type", "price_per_mw_year"]
]

,capacity_year,price_type,price_per_mw_year
2,2023,Reserve Capacity,105949.270
7,2024,Reserve Capacity,194783.535
13,2025,Reserve Capacity,251420.000
20,2026,Reserve Capacity,216091.780
30,2027,Peak Reserve Capacity,360700.000
41,2028,Peak Reserve Capacity,459191.550
